In [1]:
import pandas as pd

INPUT_PATH = "../../data/mlmart_base/v5_preprocessing.parquet"

# Ảnh chụp null TRƯỚC khi xử lý — mốc đối chiếu cho 00b_recheck_fill_null
df_raw = pd.read_parquet(INPUT_PATH)
print(f"Kích thước đầu vào: {df_raw.shape}")
null_truoc = df_raw.isna().sum()
null_truoc = null_truoc[null_truoc > 0].sort_values(ascending=False)
print("--- Null TRƯỚC xử lý (theo cột) ---")
print(null_truoc.to_string())

del df_raw  # giải phóng RAM: run_pipeline ở cell dưới sẽ tự đọc lại file

Kích thước đầu vào: (2731946, 47)
--- Null TRƯỚC xử lý (theo cột) ---
gmm_if_outlier_reason       2724515
optimizers                  1259007
capacity_kw                 1132078
number_of_panels            1132078
panel                       1132078
inverter                    1132078
site_metric                 1132078
weather_id                      218
weather_type_id                 218
weather_timestamp               218
weather_is_day                  218
shortwave_radiation             218
direct_normal_irradiance        218
diffuse_solar_radiation         218
temperature_c                   218
cloud_cover_total               218
cloud_cover_low                 218
cloud_cover_mid                 218
cloud_cover_high                218
wind_speed                      218
precipitation_mm                218
sunshine_duration               218
weather_code                    218
weather_type_is_day             218
weather_condition               218
weather_description           

In [2]:
import pandas as pd
import numpy as np

def fill_solar_site_metadata(df):
    """
    Xử lý metadata solar site theo chiến lược cụ thể:
    - capacity_kw, number_of_panels: KHÔNG fill (quyết định nhóm trưởng 2026-08-18, xem chú thích dưới).
    - panel, inverter, location_name: 'Unknown'.
    - optimizers: 'None'.
    - site_metric: 'kWh'.
    """
    # Xử lý campus_name trước (khóa nhóm theo site, giữ nguyên hành vi cũ)
    df['campus_name'] = df.groupby('site_id', observed=True)['campus_name'].transform(lambda x: x.ffill().bfill())
    df['campus_name'] = df['campus_name'].fillna('Unknown')

    # capacity_kw / number_of_panels: KHÔNG FILL (quyết định nhóm trưởng 2026-08-18):
    # 17/42 trạm null 100%; fill cũ rơi về median toàn cục 51,15 = số bịa. LightGBM xử
    # NaN native; cờ thiếu do 03_2 tạo. Hồ sơ: docs/2026_08_19_Audit_Tinh_Dung_Dan_Pipeline_V4.md

    # panel, inverter, location_name -> Unknown
    for col in ['panel', 'inverter', 'location_name']:
        if col in df.columns:
            df[col] = df.groupby('site_id', observed=True)[col].transform(lambda x: x.ffill().bfill())
            df[col] = df[col].fillna('Unknown')

    # optimizers -> None
    if 'optimizers' in df.columns:
        df['optimizers'] = df['optimizers'].fillna('None')

    # site_metric -> kWh
    if 'site_metric' in df.columns:
        df['site_metric'] = df['site_metric'].fillna('kWh')
        
    return df

def fill_geo_coordinates(df, geo_cols):
    for col in geo_cols:
        if col in df.columns:
            df[col] = df.groupby('site_id', observed=True)[col].transform(lambda x: x.ffill().bfill())
    return df

def fill_weather_metadata(df, cols):
    # weather_type_id/weather_code/weather_is_day: vestigial với ML (01 sẽ bỏ) — giữ fill để không đổi hành vi file.
    if 'weather_timestamp' in df.columns:
        df['weather_timestamp'] = pd.to_datetime(df['weather_timestamp'], errors='coerce')
        df.loc[df['weather_timestamp'].dt.year < 2019, 'weather_timestamp'] = pd.NaT

    ID_COLS = ['weather_id']
    for col in cols:
        if col not in df.columns: continue
        if col == 'weather_id': continue
        
        if col == 'weather_type_id':
            df[col] = df.groupby('site_id', observed=True)[col].transform(lambda x: x.ffill().bfill())
            df[col] = df[col].fillna(df.groupby(['month_tmp', 'hour_tmp'], observed=True)[col].transform(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan))
            df[col] = df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else np.nan)
        elif col in ['weather_condition', 'weather_description']:
            df[col] = df.groupby('site_id', observed=True)[col].transform(lambda x: x.ffill())
            df[col] = df[col].fillna('Unknown')
        elif col == 'weather_code':
            df[col] = df.groupby('site_id', observed=True)[col].transform(lambda x: x.ffill().bfill())
    return df

def fill_weather_is_day(df):
    if 'weather_is_day' not in df.columns: return df
    is_day_time = ((df['hour_tmp'] >= 6) & (df['hour_tmp'] <= 18)).astype(int)
    df['weather_is_day'] = df['weather_is_day'].fillna(is_day_time)
    
    rad_cols = ['shortwave_radiation', 'direct_normal_irradiance', 'diffuse_solar_radiation', 'sunshine_duration']
    for col in rad_cols:
        if col in df.columns:
            mask = df['weather_is_day'].isnull()
            df.loc[mask, 'weather_is_day'] = (df.loc[mask, col] > 0).astype(int)
    df['weather_is_day'] = df['weather_is_day'].fillna(0)
    return df

def fill_weather_metrics_advanced(df):
    df = df.sort_values(by=['site_id', 'timestamp']).set_index('timestamp')
    night_mask = (df['hour_tmp'] < 5.5) | (df['hour_tmp'] >= 18.5)
    
    # Đêm->0: đúng vật lý với bức xạ/sunshine; với cloud_* sai vật lý nhưng vô hại
    # (dòng đêm không vào train).
    for col in ['shortwave_radiation', 'direct_normal_irradiance', 'diffuse_solar_radiation', 'sunshine_duration',
                'cloud_cover_total', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high']:
        if col in df.columns:
            df.loc[df[col].isnull() & night_mask, col] = 0.0

    # ffill limit theo bước 15': temp 12=3h, wind 4=1h, cloud 8=2h; rồi climatology (site,hour).
    if 'temperature_c' in df.columns:
        mask = df['temperature_c'].isnull()
        df['temperature_c_is_imputed'] = mask.astype(int)
        df['temperature_c'] = df.groupby('site_id', observed=True)['temperature_c'].transform(lambda x: x.ffill(limit=12))
        df['temperature_c'] = df['temperature_c'].fillna(df.groupby(['site_id', 'hour_tmp'], observed=True)['temperature_c'].transform('median'))
        df['temperature_c'] = df['temperature_c'].fillna(df['temperature_c'].median())

    if 'wind_speed' in df.columns:
        mask = df['wind_speed'].isnull()
        df['wind_speed_is_imputed'] = mask.astype(int)
        df['wind_speed'] = df.groupby('site_id', observed=True)['wind_speed'].transform(lambda x: x.ffill(limit=4))
        df['wind_speed'] = df['wind_speed'].fillna(df.groupby(['site_id', 'hour_tmp'], observed=True)['wind_speed'].transform('median'))
        df['wind_speed'] = df['wind_speed'].fillna(df['wind_speed'].median())

    # Cloud fill
    for col in ['cloud_cover_total', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high']:
        if col in df.columns:
            mask = df[col].isnull()
            df[f'{col}_is_imputed'] = mask.astype(int)
            df[col] = df.groupby('site_id', observed=True)[col].transform(lambda x: x.ffill(limit=8))
            df[col] = df[col].fillna(df.groupby(['site_id', 'hour_tmp'], observed=True)[col].transform('median'))
            df[col] = df[col].fillna(df[col].median())

    if 'precipitation_mm' in df.columns:
        mask = df['precipitation_mm'].isnull()
        df['precipitation_mm_is_imputed'] = mask.astype(int)
        df['precipitation_mm'] = df['precipitation_mm'].fillna(0.0)

    return df.reset_index()

# Cột chữ -> category (giảm RAM). KHÔNG chuyển gmm_if_outlier_reason (01 còn fillna(""))
# và weather_join_method (01 gán nhãn mới) — category thiếu nhãn sẽ lỗi.
COT_CHU_CATEGORY = ['campus_name', 'panel', 'inverter', 'optimizers', 'site_metric',
                    'location_name', 'weather_condition', 'weather_description']

def convert_text_to_category(df):
    for col in COT_CHU_CATEGORY:
        if col in df.columns:
            df[col] = df[col].astype('category')
    return df

def run_pipeline(input_path, output_path):
    df_working = pd.read_parquet(input_path)
    
    df_working['timestamp'] = pd.to_datetime(df_working['timestamp'])
    df_working['month_tmp'] = df_working['timestamp'].dt.month
    df_working['hour_tmp'] = df_working['timestamp'].dt.hour
    
    solar_meta = ['campus_name', 'capacity_kw', 'number_of_panels', 'panel', 'inverter', 'optimizers', 'site_metric', 'location_name']
    geo_cols = ['latitude', 'longitude']
    weather_meta = ['weather_id', 'weather_type_id', 'weather_timestamp', 'weather_is_day', 'weather_code', 'weather_condition', 'weather_description']
    
    df_working = fill_geo_coordinates(df_working, geo_cols)
    df_working = fill_solar_site_metadata(df_working)
    df_working = fill_weather_metadata(df_working, weather_meta)
    df_working = fill_weather_is_day(df_working)
    df_working = fill_weather_metrics_advanced(df_working)

    _truoc_mb = df_working.memory_usage(deep=True).sum() / 1024**2
    df_working = convert_text_to_category(df_working)
    _sau_mb = df_working.memory_usage(deep=True).sum() / 1024**2
    print(f"RAM sau khi chuyển {len(COT_CHU_CATEGORY)} cột chữ sang category: "
          f"{_truoc_mb:.1f} MB -> {_sau_mb:.1f} MB (giảm {(1 - _sau_mb / _truoc_mb) * 100:.1f}%)")
    
    df_working = df_working.drop(columns=['month_tmp', 'hour_tmp'])
    df_working.to_parquet(output_path, index=False)
    return df_working

print("Đã nạp hàm fill null và sẵn sàng chạy pipeline.")


Đã nạp hàm fill null và sẵn sàng chạy pipeline.


In [3]:
INPUT_PATH = "../../data/mlmart_base/v5_preprocessing.parquet"
OUTPUT_PATH = "../../data/mlmart_base/v5_final_cleaned.parquet"
df_clean = run_pipeline(INPUT_PATH, OUTPUT_PATH)

# Null CÓ CHỦ ĐÍCH (PASS không còn nghĩa là "0 null") — lý do từng cột: xem cell trên.
CHO_PHEP_NULL = {'capacity_kw', 'number_of_panels', 'gmm_if_outlier_reason',
                 'weather_id', 'weather_timestamp', 'weather_type_is_day'}
null_sau = df_clean.isna().sum()
null_sau = null_sau[null_sau > 0].sort_values(ascending=False)
ngoai_du_kien = null_sau[~null_sau.index.isin(CHO_PHEP_NULL)]
status = "PASS" if ngoai_du_kien.empty else "FAIL"

summary = pd.DataFrame([
    {
        "stage": "00_fill_null_imputation",
        "input": INPUT_PATH,
        "output": OUTPUT_PATH,
        "rows": len(df_clean),
        "columns": len(df_clean.columns),
        "null_values": int(df_clean.isna().sum().sum()),
        "status": status,
    }
])
display(summary)
print("--- Null còn lại theo cột (kỳ vọng CHỈ nằm trong CHO_PHEP_NULL) ---")
print(null_sau.to_string())
if not ngoai_du_kien.empty:
    print("\n[FAIL] Cột null NGOÀI dự kiến — kiểm lại trước khi đi tiếp:")
    print(ngoai_du_kien.to_string())
print("da chay")

RAM sau khi chuyển 8 cột chữ sang category: 1618.1 MB -> 1082.0 MB (giảm 33.1%)


,stage,input,output,rows,columns,null_values,status
0,00_fill_null_imputation,../../data/mlmart_base/v5_preprocessing.parquet,../../data/mlmart_base/v5_final_cleaned.parquet,2731946,54,4989325,PASS


--- Null còn lại theo cột (kỳ vọng CHỈ nằm trong CHO_PHEP_NULL) ---
gmm_if_outlier_reason    2724515
capacity_kw              1132078
number_of_panels         1132078
weather_id                   218
weather_timestamp            218
weather_type_is_day          218
da chay


In [4]:
def list_capacity_by_campus(df, campus_col="campus_name", capacity_col="capacity_kw"):
    """Danh sách capacity duy nhất của từng campus."""
    return (
        df.groupby(campus_col, observed=True)[capacity_col]
          .agg(lambda s: sorted(s.dropna().unique()))
          .rename("capacity_list")
          .reset_index()
    )

In [5]:
# Sau khi bỏ fill: 4 campus toàn-null kỳ vọng ra list RỖNG (bản cũ ra [51.15] — số bịa)
list_capacity_by_campus(df_clean)

,campus_name,capacity_list
0,Albury-Wodonga,[]
1,Bendigo,[]
2,Bundoora,"[21.39, 29.45, 30.0, 34.32, 35.64, 38.94, 39.2..."
3,Mildura,[]
4,Shepparton,[]


In [6]:
# ── Bằng chứng sau khi bỏ fill capacity/panels ──
# Kỳ vọng: 17/42 trạm NaN (Albury-Wodonga 5, Bendigo 8, Bundoora 2 [site 28, 29],
# Mildura 1, Shepparton 1); số dòng null mỗi cột = 1.132.078.
kiem = (df_clean.groupby(['campus_name', 'site_id'], observed=True)['capacity_kw']
        .first().reset_index())
n_nan = int(kiem['capacity_kw'].isna().sum())
print(f"Số trạm capacity_kw = NaN     : {n_nan}/{kiem['site_id'].nunique()} (kỳ vọng 17/42)")
print(f"Số dòng null capacity_kw      : {int(df_clean['capacity_kw'].isna().sum()):,} (kỳ vọng 1,132,078)")
print(f"Số dòng null number_of_panels : {int(df_clean['number_of_panels'].isna().sum()):,} (kỳ vọng bằng capacity_kw)")
kiem

Số trạm capacity_kw = NaN     : 17/42 (kỳ vọng 17/42)
Số dòng null capacity_kw      : 1,132,078 (kỳ vọng 1,132,078)
Số dòng null number_of_panels : 1,132,078 (kỳ vọng bằng capacity_kw)


,campus_name,site_id,capacity_kw
0,Albury-Wodonga,1,NaN
1,Albury-Wodonga,2,NaN
2,Albury-Wodonga,3,NaN
3,Albury-Wodonga,4,NaN
4,Albury-Wodonga,5,NaN
5,Bendigo,6,NaN
6,Bendigo,7,NaN
7,Bendigo,8,NaN
8,Bendigo,9,NaN
9,Bendigo,10,NaN


In [7]:
import pandas as pd
import numpy as np

def compare_imputation(df_raw, df_clean):
    """
    So sánh giá trị Null trước/sau và chỉ tính thống kê trên các cột Metrics.
    """
    # Danh sách Metrics cần tính thống kê
    metric_cols = [
        'shortwave_radiation', 'direct_normal_irradiance', 'diffuse_solar_radiation',
        'temperature_c', 'cloud_cover_total', 'cloud_cover_low', 'cloud_cover_mid', 
        'cloud_cover_high', 'wind_speed', 'precipitation_mm', 'sunshine_duration'
    ]
    
    # Chỉ lấy những cột thực sự tồn tại trong DataFrame
    common_metrics = [c for c in metric_cols if c in df_raw.columns and c in df_clean.columns]
    
    # 1. Tổng kết Null (áp dụng cho tất cả cột để kiểm tra coverage)
    all_cols = [c for c in df_raw.columns if c in df_clean.columns]
    summary = pd.DataFrame({
        'Null_Before': df_raw[all_cols].isnull().sum(),
        'Null_After': df_clean[all_cols].isnull().sum()
    })
    summary['Filled_Count'] = summary['Null_Before'] - summary['Null_After']
    
    print("--- TỔNG KẾT SỐ LƯỢNG GIÁ TRỊ ĐÃ FILL ---")
    print(summary[summary['Filled_Count'] > 0].to_string())
    
    # 2. So sánh Mean chỉ trên các cột Metrics
    print("\n--- SO SÁNH MEAN (CHỈ CÁC CỘT METRIC) ---")
    stats_compare = pd.DataFrame({
        'Mean_Before': df_raw[common_metrics].mean(numeric_only=True),
        'Mean_After': df_clean[common_metrics].mean(numeric_only=True)
    })
    # Thêm cột % chênh lệch để dễ đánh giá độ lệch phân phối
    stats_compare['Diff_%'] = ((stats_compare['Mean_After'] - stats_compare['Mean_Before']) / stats_compare['Mean_Before'] * 100).abs()
    print(stats_compare.to_string())
    
    # 3. Mẫu thay đổi
    print("\n--- MẪU DỮ LIỆU ĐÃ THAY ĐỔI ---")
    for col in common_metrics:
        if summary.loc[col, 'Filled_Count'] > 0:
            mask = df_raw[col].isnull() & df_clean[col].notnull()
            if mask.any():
                print(f"\nCột: {col}")
                sample = pd.concat([df_raw.loc[mask, col].head(3), df_clean.loc[mask, col].head(3)], axis=1)
                sample.columns = ['Before (NaN)', 'After (Filled)']
                print(sample)

# Audit theo row-group để không giữ đồng thời hai DataFrame 2.7 triệu dòng trong RAM.
import pyarrow.compute as pc
import pyarrow.parquet as pq

raw_file = pq.ParquetFile("../../data/mlmart_base/v5_preprocessing.parquet")
clean_file = pq.ParquetFile("../../data/mlmart_base/v5_final_cleaned.parquet")
common_cols = [c for c in raw_file.schema_arrow.names if c in clean_file.schema_arrow.names]
metric_cols = [c for c in [
    'shortwave_radiation', 'direct_normal_irradiance', 'diffuse_solar_radiation',
    'temperature_c', 'cloud_cover_total', 'cloud_cover_low', 'cloud_cover_mid',
    'cloud_cover_high', 'wind_speed', 'precipitation_mm', 'sunshine_duration'
] if c in common_cols]

def streaming_stats(parquet_file, columns):
    nulls = {c: 0 for c in columns}
    sums = {c: 0.0 for c in metric_cols}
    counts = {c: 0 for c in metric_cols}
    for batch in parquet_file.iter_batches(batch_size=100_000, columns=columns):
        for name, array in zip(batch.schema.names, batch.columns):
            nulls[name] += array.null_count
            if name in metric_cols:
                value = pc.sum(array).as_py()
                sums[name] += float(value or 0.0)
                counts[name] += len(array) - array.null_count
    means = {c: sums[c] / counts[c] if counts[c] else np.nan for c in metric_cols}
    return nulls, means

raw_nulls, raw_means = streaming_stats(raw_file, common_cols)
clean_nulls, clean_means = streaming_stats(clean_file, common_cols)
summary = pd.DataFrame({'Null_Before': raw_nulls, 'Null_After': clean_nulls})
summary['Filled_Count'] = summary['Null_Before'] - summary['Null_After']
print('--- TỔNG KẾT SỐ LƯỢNG GIÁ TRỊ ĐÃ FILL ---')
print(summary[summary['Filled_Count'] > 0].to_string())
stats_compare = pd.DataFrame({'Mean_Before': raw_means, 'Mean_After': clean_means})
stats_compare['Diff_%'] = ((stats_compare['Mean_After'] - stats_compare['Mean_Before']) / stats_compare['Mean_Before'] * 100).abs()
print('\n--- SO SÁNH MEAN (CHỈ CÁC CỘT METRIC) ---')
print(stats_compare.to_string())

--- TỔNG KẾT SỐ LƯỢNG GIÁ TRỊ ĐÃ FILL ---
                          Null_Before  Null_After  Filled_Count
panel                         1132078           0       1132078
inverter                      1132078           0       1132078
optimizers                    1259007           0       1259007
site_metric                   1132078           0       1132078
weather_type_id                   218           0           218
weather_is_day                    218           0           218
shortwave_radiation               218           0           218
direct_normal_irradiance          218           0           218
diffuse_solar_radiation           218           0           218
temperature_c                     218           0           218
cloud_cover_total                 218           0           218
cloud_cover_low                   218           0           218
cloud_cover_mid                   218           0           218
cloud_cover_high                  218           0           21